# QSVT as Matrix Functions: Powers and Roots

QSVT implements polynomial transformations of eigenvalues.

By approximating a function f(x) with a bounded polynomial P(x),
QSVT implements:

    P(A) ≈ f(A)

In this notebook we explore:

- matrix powers
- square roots
- fractional powers

using small matrices to build intuition.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qsvt.matrices import rotation

from qsvt.spectral import (
    eigh_hermitian,
    apply_function_to_hermitian,
    matrix_power_spectral,
    matrix_square_root,
    matrix_fractional_power,
)

from qsvt.polynomials import (
    eval_polynomial,
)

## 1. Test matrix

In [ ]:
theta = 0.6

U = rotation(theta)

lambdas = np.array([0.2, 0.8])

A = U @ np.diag(lambdas) @ U.T

eigvals, eigvecs = eigh_hermitian(A)

print("A =")
print(np.round(A, 6))

print("\nEigenvalues =", eigvals)

## 2. Integer powers

In [ ]:
powers = [1, 2, 3, 4]

plt.figure(figsize=(7, 5))

for k in powers:

    eigvals_power = eigvals**k

    plt.plot(eigvals, eigvals_power, "o-", label=f"x^{k}")

plt.xlabel("original eigenvalue")
plt.ylabel("transformed eigenvalue")

plt.title("Spectral effect of matrix powers")

plt.grid(True)
plt.legend()

plt.show()

In [ ]:
A2 = matrix_power_spectral(A, 2)

print("A^2 via spectral map:")
print(np.round(A2, 6))

## 3. Polynomial approximation to sqrt(x)

In [ ]:
a = 0.2
deg = 6


def sqrt_fn(x):
    return np.sqrt(x)


def chebyshev_fit_on_interval(func, lower, upper, degree, num_points=500):
    """
    Fit a polynomial to func on [lower, upper] using a Chebyshev-basis fit,
    then return standard monomial coefficients in ascending degree order.
    """
    xs = np.linspace(lower, upper, num_points)
    ys = np.asarray(func(xs), dtype=float)

    cheb_coeffs = np.polynomial.chebyshev.chebfit(xs, ys, degree)
    poly = np.polynomial.Chebyshev(cheb_coeffs, domain=[-1.0, 1.0])
    coeffs = poly.convert(kind=np.polynomial.Polynomial).coef

    return np.asarray(coeffs, dtype=float)


coeffs_sqrt = chebyshev_fit_on_interval(
    sqrt_fn,
    lower=a,
    upper=1.0,
    degree=deg,
)

x_fit = np.linspace(a, 1.0, 2001)

print(
    "bounded on [a,1]:",
    bool(np.max(np.abs(eval_polynomial(coeffs_sqrt, x_fit))) <= 1.0 + 1e-8),
)

In [ ]:
xs = np.linspace(a, 1.0, 400)

plt.figure(figsize=(7, 5))
plt.plot(xs, sqrt_fn(xs), "--", label=r"$\sqrt{x}$ (exact)")
plt.plot(xs, eval_polynomial(coeffs_sqrt, xs), label=f"Polynomial approx (deg={deg})")
plt.xlabel("x")
plt.ylabel("value")
plt.title("Polynomial Approximation to sqrt(x) on [a,1]")
plt.grid(True)
plt.legend()
plt.show()

## 4. Applying sqrt(A)

In [ ]:
sqrtA_exact = matrix_square_root(A)

sqrtA_poly = apply_function_to_hermitian(A, lambda x: eval_polynomial(coeffs_sqrt, x))

print("sqrt(A) exact:")
print(np.round(sqrtA_exact, 6))

print("\nsqrt(A) polynomial:")
print(np.round(sqrtA_poly, 6))

## 5. Fractional powers

In [ ]:
alphas = [0.25, 0.5, 0.75]

plt.figure(figsize=(7, 5))

for alpha in alphas:

    plt.plot(eigvals, eigvals**alpha, "o-", label=f"x^{alpha}")

plt.xlabel("original eigenvalue")
plt.ylabel("transformed eigenvalue")

plt.title("Fractional spectral maps")

plt.grid(True)
plt.legend()

plt.show()

In [ ]:
A_half = matrix_fractional_power(A, 0.5)

print("A^0.5 via spectral routine:")
print(np.round(A_half, 6))

## Summary

Matrix functions act spectrally:

    f(A) = U f(Λ) U†

QSVT implements polynomial approximations to these maps.

Powers, roots, and fractional transforms are all accessible
through bounded polynomial design.

## Validation

Compact checks for the expected tutorial behavior.


In [ ]:
assert np.allclose(A2, A @ A, atol=1e-10)
assert np.allclose(sqrtA_exact @ sqrtA_exact, A, atol=1e-10)
assert np.allclose(sqrtA_poly, sqrtA_exact, atol=1e-3)
assert np.allclose(A_half, sqrtA_exact, atol=1e-10)

print(f"sqrt_poly_max_error: {np.max(np.abs(sqrtA_poly - sqrtA_exact)):.3e}")
print(f"spectral_square_error: {np.max(np.abs(A2 - A @ A)):.3e}")
print("validation: passed")